<a href="https://colab.research.google.com/github/Jolayemi-momoh/sexy-ribbon/blob/main/my_dear_watson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import kagglehub
path = '/content/drive/MyDrive/contradictory-my-dear-watson'
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import os

# List the contents of the downloaded data directory
print(os.listdir(path))

In [ ]:
!pip install -q keras-nlp --upgrade
!pip install seaborn

In [ ]:
pip install --upgrade pip

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import os

In [ ]:
import keras_nlp
print('tensorflow:', tf.__version__)
print('KerasNLP version', keras_nlp.__version__)

##Accelerator

In [ ]:
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
    strategy= tf.distribute.TPUStrategy(tpu)
except ValueError:
    print('TPU not activated')
    strategy= tf.distribute.MirroredStrategy()
print('replicas:',strategy.num_replicas_in_sync)

In [ ]:
DATA_DIR = path

In [ ]:
RESULT_DICT = {
    0 : "entailment",
    1 : "neutral",
    2 : "contradiction"
}

In [ ]:
for dirname, _, filenames in os.walk(DATA_DIR):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
df_train=pd.read_csv(DATA_DIR+'/train.csv')
df_test=pd.read_csv(DATA_DIR+'/test.csv')

In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
def display_pair_of_sentence(x):
  print( "Premise : " + x[ 'premise' ])
  print('Hypothesis: '+x[ 'hypothesis' ])
  print( 'Language: '+ x[ 'language' ])
  print('Label: '+ str(x[ 'label']))
  print()

  df_train.head(10).apply(lambda x: display_pair_of_sentence(x), axis=1)

In [ ]:
df_train.shape

In [ ]:
for index, row in df_train.head(10).iterrows():
    print("Premise : " + row['premise'])
    print('Hypothesis: ' + row['hypothesis'])
    print('Language: ' + row['language'])
    print('Label: ' + str(row['label']))
    print()

In [ ]:
f,ax= plt.subplots(figsize=(12, 4))
sns.set_color_codes("pastel")
sns.despine()
ax=sns.countplot(data=df_train, y="label",
                 order=df_train['label'].value_counts().index)
abs_values=df_train['label'].value_counts(ascending=False)
rel_values=df_train['label'].value_counts(ascending=False, normalize=True).values*100
lbls=[f'{p[0]} ({p[1]:.0f}%)' for p in zip(abs_values.index, rel_values)]
ax.bar_label(container=ax.containers[0],labels=lbls)
ax


ax.set_xlabel("Count")
ax.set_ylabel("Label")

In [ ]:
f,ax= plt.subplots(figsize=(12, 4))
sns.set_color_codes("pastel")
sns.despine()
ax=sns.countplot(data=df_train,
                y='language',
                order=df_train['language'].value_counts().index)
abs_values=df_train['language'].value_counts(ascending=False)
rel_values=df_train['language'].value_counts(ascending=False,normalize=True).values*100
lbls=[f'{p[0]} ({p[1]:0f}%)' for p in zip(abs_values, rel_values)]
ax.bar_label(ax.containers[0], labels=lbls)
ax.set_title('Distribution of languages in the trainning set')


In [ ]:
df_train['premise_length']=df_train['premise'].apply(lambda x:len(x))
df_train['hypothesis_length']=df_train['hypothesis'].apply(lambda x:len(x))
df_train[['hypothesis_length','premise_length']].describe()

In [ ]:
VALIDATION_SPLIT=0.3
TRAIN_SIZE=int(df_train.shape[0]*(1-VALIDATION_SPLIT))
BATCH_SIZE=8*strategy.num_replicas_in_sync

In [ ]:
def split_labels(x,y):
  return(x[0],x[1]),y

training_dataset=(
    tf.data.Dataset.from_tensor_slices(
        (
            (df_train[['premise','hypothesis']].values),
            keras.utils.to_categorical(df_train['label'],num_classes=3)
        )
    )
)

train_dataset=training_dataset.take(TRAIN_SIZE)
val_dataset=training_dataset.skip(TRAIN_SIZE)
train_preprocessed=train_dataset.map(split_labels,tf.data.AUTOTUNE).batch(BATCH_SIZE,drop_remainder=True).cache().prefetch(tf.data.AUTOTUNE)
val_preprocessed=val_dataset.map(split_labels,tf.data.AUTOTUNE).batch(BATCH_SIZE,drop_remainder=True).cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
with strategy.scope():
  classifier = keras_nlp.models.BertClassifier.from_preset("bert_base_multi",num_classes=3)
  classifier.compile(optimizer=tf.keras.optimizers.Adam(1e-5*strategy.num_replicas_in_sync),
                                                  loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                                                  metrics=['accuracy'])

In [ ]:
classifier.summary()

In [ ]:
import torch
print(torch.cuda.is_available())
EPOCHS=3
with strategy.scope():
  history = classifier.fit(train_preprocessed,
                           epochs=EPOCHS,
                           validation_data=val_preprocessed
                          )